# Grouped Planet Model — Adım Adım İşleyiş

Bu notebook `model_grouped_planet.py` içindeki **GroupedPlanetOrbitWarsPolicy**'nin bir turn içinde nasıl karar verdiğini, her ara tensor'un shape ve anlamını göstererek anlatır.

Mimari özet:

1. `PlanetProjector` — planet feature vektörünü tek `d_model` token'a yansıtır (group ayrımı yok).
2. `GroupedPlanetRelationEncoder` — P planet token üzerinde transformer (full / induced) attention.
   `full` modda planet-planet edge bias attention'a eklenir.
3. PMA pool — küçük sayıda learnable seed cross-attention ile global summary üretir.
4. Heads:
   - `src_head` — per-source `source_launch[s]` skoru
   - `noop_head` — per-source NOOP raw skoru (en yükseği global NOOP slot olarak da eklenir)
   - `tgt_query / tgt_key` — bilinear pairwise pointer (P×P)
   - **`ship_head` — pair-conditioned `Beta(α, β)` parametreleri (P×P×2)**, sample → ship_value ∈ [0,1] → `ship_value × SHIP_REQUEST_CAP` istenen gemi
   - `value_head`, `aux_owner_head`
5. Rollout / watch — turn başına en fazla 8 launch, her karar adımında en iyi 64 source-target çifti.

> **Not (2026-05):** Ship head'i eskiden 5 ayrık bin'di (`ship_logits`). Continuous Beta head'e geçildi:
> sürekli kontrol + tek skaler `log_prob` + garrison aşımı için `overflow_penalty`. Eski `ship_bin`
> isimleri yerine artık `ship_value`, `ship_params` kullanılır.

In [1]:
import os, sys, pathlib
repo_root = pathlib.Path().resolve().parent if pathlib.Path.cwd().name == 'docs' else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
os.chdir(repo_root)
print('repo_root =', repo_root)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
torch.manual_seed(0)
np.random.seed(0)
print('torch', torch.__version__)

repo_root = /home/doruk/Documents/Kaggle/Orbit_Wars


torch 2.11.0+cu128


## 1. Observation üretimi

Kaggle env'inden bir step alıyoruz, `build_observation` ham gözlemi modelin beklediği numpy tensorlarına çevirir.

Önemli alanlar:

- `planet_feats (P_MAX, PLANET_FEATURE_DIM)`
- `planet_mask (P_MAX,)`
- `source_mask (P_MAX,)`
- `global_feats (GLOBAL_FEATURE_DIM,)`
- `edge_bias (P+F+1, P+F+1, EDGE_FEATURE_DIM)`

In [2]:
from kaggle_environments import make
from obs_builder import build_observation, PLANET_FEATURE_DIM, GLOBAL_FEATURE_DIM, EDGE_FEATURE_DIM, P_MAX_DEFAULT

env = make('orbit_wars', debug=False)
env.configuration['randomSeed'] = 42
state = env.reset(num_agents=2)
state = env.step([[], []])
raw_obs = state[0]['observation']
omega = raw_obs.get('angular_velocity', 0.0) or 0.0
obs = build_observation(raw_obs, player=0, angular_velocity=omega)

print('P_MAX             =', P_MAX_DEFAULT)
print('PLANET_FEATURE_DIM=', PLANET_FEATURE_DIM)
print('GLOBAL_FEATURE_DIM=', GLOBAL_FEATURE_DIM)
print('EDGE_FEATURE_DIM  =', EDGE_FEATURE_DIM)
print()
print('planet_feats shape =', obs.planet_feats.shape)
print('planet_mask  shape =', obs.planet_mask.shape, '  active =', int(obs.planet_mask.sum()))
print('source_mask  shape =', obs.source_mask.shape, '  sources=', int(obs.source_mask.sum()))
print('edge_bias    shape =', obs.edge_bias.shape)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy


... (10 outputs left out)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_python_repeated_pokerkit


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: OpenSpiel games skipped: 1.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    snake


P_MAX             = 44
PLANET_FEATURE_DIM= 34
GLOBAL_FEATURE_DIM= 14
EDGE_FEATURE_DIM  = 6

planet_feats shape = (44, 34)
planet_mask  shape = (44,)   active = 24
source_mask  shape = (44,)   sources= 1
edge_bias    shape = (109, 109, 6)


## 2. `PlanetProjector` — planet feature → tek token

Her planet'in feature vektörü iki katlı bir MLP ile tek bir `d_model` boyutlu token'a yansıtılır. Group token / group embedding yoktur; özellikler arası tüm etkileşim attention layerlarına bırakılmıştır.

Çıktı shape: `(B, P, d_model)`.

In [3]:
from model_grouped_planet import PlanetProjector, GroupedPlanetOrbitWarsPolicy

D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2

proj = PlanetProjector(planet_dim=PLANET_FEATURE_DIM, d_model=D_MODEL)
n_params = sum(p.numel() for p in proj.parameters())
print('PlanetProjector params:', n_params)

pf = torch.from_numpy(obs.planet_feats).unsqueeze(0).float()
tok = proj(pf)
print('planet_feats   ->', tuple(pf.shape))
print('planet token   ->', tuple(tok.shape), '  (B, P, d_model)')
print('mean / std     ->', f'{tok.mean().item():+.3f} / {tok.std().item():.3f}')

PlanetProjector params: 6528
planet_feats   -> (1, 44, 34)
planet token   -> (1, 44, 64)   (B, P, d_model)
mean / std     -> +0.000 / 0.997


## 3. Encoder — P-token attention + PMA pool

Tek bir token-per-planet ile attention bloklarına gireriz:

- `full` modda standart Transformer block, planet-planet edge bias attention skorlarına eklenir.
- `induced` modda ISAB; learnable inducing points üzerinden O(P·M) maliyetli attention, edge bias yok.

Pooling: `n_pool_seeds` adet learnable seed ile cross-attention; pooled token'lar fuze edilip global summary `(B, d_model)` üretilir.

In [4]:
policy = GroupedPlanetOrbitWarsPolicy(
    p_max=P_MAX_DEFAULT,
    planet_dim=PLANET_FEATURE_DIM,
    global_dim=GLOBAL_FEATURE_DIM,
    edge_dim=EDGE_FEATURE_DIM,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    attention_type='induced',
    n_inducing_points=8,
)
policy.eval()
enc = policy.encoder
print('attention_type   =', enc.attention_type)
print('n_pool_seeds     =', enc.n_pool_seeds)
print('pool_seeds shape =', tuple(enc.pool_seeds.shape))
print('total policy params =', sum(p.numel() for p in policy.parameters()))

attention_type   = induced
n_pool_seeds     = 4
pool_seeds shape = (4, 64)
total policy params = 293222


### 3.1 Tokenization, padding mask, edge bias

In [5]:
from model import pack_observations

batch = pack_observations([obs], device=torch.device('cpu'))
print('batch keys:', list(batch.keys()))

x = enc._build_tokens(batch['planet_feats'], batch['fleet_feats'], batch['global_feats'])
print('encoder input tokens ->', tuple(x.shape), '  (B, P, d_model)')

kpm = enc._build_padding_mask(batch['planet_mask'], batch['fleet_mask'])
print('padding mask shape   ->', tuple(kpm.shape), '  active=', int((~kpm).sum().item()))

if enc.attention_type == 'full':
    bias = enc._build_attn_bias(batch['edge_bias'])
    print('attn_bias shape      ->', tuple(bias.shape), '  (B, H, P, P)')
else:
    print('attn_bias            ->  (induced mode kullanmıyor)')

batch keys: ['planet_feats', 'fleet_feats', 'global_feats', 'edge_bias', 'planet_mask', 'fleet_mask', 'source_mask']
encoder input tokens -> (1, 44, 64)   (B, P, d_model)
padding mask shape   -> (1, 44)   active= 24
attn_bias            ->  (induced mode kullanmıyor)


### 3.2 `encode_planets` çıktıları

In [6]:
with torch.no_grad():
    enc_out = enc.encode_planets(
        planet_feats=batch['planet_feats'],
        fleet_feats=batch['fleet_feats'],
        global_feats=batch['global_feats'],
        edge_bias=batch['edge_bias'],
        planet_mask=batch['planet_mask'],
        fleet_mask=batch['fleet_mask'],
    )
for k, v in enc_out.items():
    if isinstance(v, torch.Tensor):
        print(f'{k:14s} -> shape={tuple(v.shape)}  mean={v.float().mean().item():+.3f}  std={v.float().std().item():.3f}')

planet_tokens  -> shape=(1, 44, 64)  mean=+0.000  std=0.739
pool_seeds     -> shape=(1, 4, 64)  mean=-0.000  std=1.002
global_summary -> shape=(1, 64)  mean=+0.000  std=1.008


## 4. Policy heads

Encoder çıktıları planet token + global summary'dir. Heads:

| Head | Çıktı | Shape | Anlam |
|---|---|---|---|
| `src_head` | per-source launch score | (B, P) | her source'un launch eğilimi `ℓ_s` |
| `noop_head` | per-source noop raw + global noop | (B, P+1) | son slot global NOOP (per-source noop_raw'ın max'ı) |
| `tgt_query / tgt_key` | bilinear pairwise pointer | (B, P, P) | `tgt_logits[s,t] = (q_s · k_t)/√d + ℓ_s` |
| **`ship_head`** | **pair-conditioned Beta params** | **(B, P, P, 2)** | **(α, β) → Beta(α,β); sample = ship_value ∈ [0,1]** |
| `value_head` | state value | (B,) | PPO critic |
| `aux_owner_head` | owner aux | (B, P, 5) | yardımcı self-supervision logiti |

`src_logits` final shape (B, P+1): ilk P slot per-source `noop_raw[s]` (kategorik içinde NOOP-vs-launch), son slot global NOOP.

In [7]:
with torch.no_grad():
    out = policy(**batch)

print('src_logits        ->', tuple(out.src_logits.shape), '  (P+1: son slot global no-op)')
print('tgt_logits        ->', tuple(out.tgt_logits.shape), '  (B, P, P)  bilinear pair scores')
print('ship_params       ->', tuple(out.ship_params.shape), '  (B, P, P, 2) = (alpha, beta) Beta head')
print('value             ->', tuple(out.value.shape))
print('aux_owner_logits  ->', tuple(out.aux_owner_logits.shape))
print('tokens (planet)   ->', tuple(out.tokens.shape), '  (B, P, d_model)')

src_logits        -> (1, 45)   (P+1: son slot global no-op)
tgt_logits        -> (1, 44, 44)   (B, P, P)  bilinear pair scores
ship_params       -> (1, 44, 44, 2)   (B, P, P, 2) = (alpha, beta) Beta head
value             -> (1,)
aux_owner_logits  -> (1, 44, 5)
tokens (planet)   -> (1, 44, 64)   (B, P, d_model)


### 4.1 `tgt_logits` — bir source'un en yüksek 5 hedefi

In [8]:
src_slots = [int(i) for i, m in enumerate(obs.source_mask) if m]
print('source slotlar:', src_slots)

if src_slots:
    s = src_slots[0]
    top_targets = torch.topk(out.tgt_logits[0, s], k=5)
    print(f'\nsource_slot={s} en yüksek 5 target:')
    for val, idx in zip(top_targets.values.tolist(), top_targets.indices.tolist()):
        print(f'  target_slot={idx:3d}  logit={val:+.3f}  planet_id={int(obs.planet_ids[idx])}')

source slotlar: [12]

source_slot=12 en yüksek 5 target:
  target_slot= 19  logit=+0.358  planet_id=19
  target_slot= 18  logit=+0.312  planet_id=18
  target_slot= 11  logit=+0.286  planet_id=11
  target_slot=  7  logit=+0.272  planet_id=7
  target_slot= 17  logit=+0.269  planet_id=17


### 4.2 `ship_params` — pair-conditioned Beta(α, β)

Her `(src, tgt)` çifti için iki sayı: `(α, β)`. Bunlar `_beta_concentrations` ile pozitif aralığa
mapleniyor: önce `softplus`, sonra `+1` (taban `α, β ≥ 1` ⇒ unimodal Beta), sonra **tanh soft-cap
ile [1, 30] aralığına sıkıştırılıyor** (delta'ya çöküşü ve PPO ratio explosion'ını önler).

Sample: `ship_value ~ Beta(α, β) ∈ (0, 1)`. Decode: `requested_ships = ship_value × SHIP_REQUEST_CAP`.

In [9]:
from model import _beta_concentrations, _make_ship_dist
from obs_builder import SHIP_REQUEST_CAP

if src_slots:
    s = src_slots[0]
    print(f'src_slot={s}, top 3 target için Beta(α, β) ve örnekler:')
    for t in top_targets.indices.tolist()[:3]:
        raw = out.ship_params[0, s, t]              # (2,) raw α, β logits
        alpha, beta = _beta_concentrations(raw)
        dist = _make_ship_dist(raw)
        mean = (alpha / (alpha + beta)).item()
        sample = dist.sample().item()
        print(f'  tgt={t:3d}  α={alpha.item():.2f} β={beta.item():.2f}  '
              f'mean={mean:.3f}  sample={sample:.3f}  '
              f'requested_ships≈{sample * SHIP_REQUEST_CAP:.0f}')

src_slot=12, top 3 target için Beta(α, β) ve örnekler:
  tgt= 19  α=1.69 β=1.60  mean=0.513  sample=0.512  requested_ships≈102
  tgt= 18  α=1.69 β=1.61  mean=0.511  sample=0.279  requested_ships≈56
  tgt= 11  α=1.69 β=1.59  mean=0.515  sample=0.134  requested_ships≈27


## 5. `act` — joint source-target + Beta ship_value

Grouped policy `act`'ı NOOP + tüm source-target çiftleri üzerinde tek bir Categorical kurar
(P² + 1 alternatif). Seçilen `(src, tgt)` çifti için Beta head'inden `ship_value` örneklenir.

Output:
- `src`        : seçilen source slot (long)
- `tgt`        : seçilen target slot (long)
- `ship_value` : Beta(α, β) sample ∈ [0, 1] (float)
- `is_noop`    : NOOP slotu seçildi mi (bool)
- `log_prob`   : `log p(action) + 1[¬noop] · log p_Beta(ship_value)`

In [10]:
with torch.no_grad():
    action = policy.act(out, deterministic=True)
for k, v in action.items():
    if isinstance(v, torch.Tensor):
        # ship_value'yu float olarak yazdır, diğerleri long/bool
        if k == 'ship_value':
            print(f'{k:10s} -> {[round(x, 4) for x in v.tolist()]}')
        else:
            print(f'{k:10s} -> {v.tolist()}')

src        -> [12]
tgt        -> [19]
ship_value -> [0.5134]
is_noop    -> [False]
log_prob   -> [-2.686636447906494]


## 6. `act_source` — fixed source

Rollout aynı turn içinde aynı source'tan farklı target'lara launch edebilmek için `act_source` kullanır:
source slot dışarıdan verilir; sadece **NOOP-vs-launch + target seçimi + ship_value** kurulur.

İç yapı (per source `s`):
```
action_logits = [ noop_raw[s],
                  qk[s, t_0] + ℓ_s,
                  qk[s, t_1] + ℓ_s,
                  ...,
                  qk[s, t_{P-1}] + ℓ_s ]   # length P+1
action_idx ~ Categorical(softmax(action_logits))
ship_value ~ Beta(α[s, tgt], β[s, tgt])    # (eğer launch seçildiyse)
```

In [11]:
if src_slots:
    s = src_slots[0]
    src_tensor = torch.tensor([s], dtype=torch.long)
    with torch.no_grad():
        a = policy.act_source(out, source_slot=src_tensor, deterministic=True)
    print(f'source={s} için deterministic action:')
    for k, v in a.items():
        if isinstance(v, torch.Tensor):
            if k == 'ship_value':
                print(f'  {k:10s} -> {[round(x, 4) for x in v.tolist()]}')
            else:
                print(f'  {k:10s} -> {v.tolist()}')

source=12 için deterministic action:
  src        -> [12]
  tgt        -> [19]
  ship_value -> [0.5134]
  is_noop    -> [False]
  log_prob   -> [-2.686636447906494]


## 7. `evaluate_source_actions` — PPO log-prob/entropy

PPO update'inde rollout'ta saklı `(src, tgt, ship_value, is_noop)` için log-prob ve entropy yeniden
hesaplanır; clipped surrogate ratio = exp(new_logp − old_logp) bunun üzerinden kurulur.

**Önemli:** `ship_value` artık **float** (eski `ship_bin` long değil). Beta log-prob'u da
diferansiyel entropi içerir; total entropy formülü:

```
H = H(action) + Pr(launch) · H_Beta(α, β)
```

`Pr(launch)` mevcut policy'nin marjinal launch olasılığı (eski koddaki realize edilmiş `valid` flag yerine).
Bu, launch oranı PPO epoch'ları içinde kayarken entropy gradient'ini bias'sız tutuyor.

In [12]:
if src_slots:
    s = src_slots[0]
    src_t = torch.tensor([s], dtype=torch.long)
    tgt_t = torch.tensor([int(top_targets.indices[0])], dtype=torch.long)
    # Continuous Beta head: ship_value float ∈ [0, 1]
    ship_value_t = torch.tensor([0.5], dtype=torch.float32)
    noop_t = torch.tensor([False], dtype=torch.bool)
    ev = policy.evaluate_source_actions(out, src_t, tgt_t, ship_value_t, noop_t)
    for k, v in ev.items():
        if isinstance(v, torch.Tensor):
            print(f'{k:10s} -> {v.tolist()}')

log_prob   -> [-2.688539505004883]
entropy    -> [3.1013753414154053]
value      -> [0.1989654004573822]


## 8. Rollout — ranked source-target seçimi + Beta sample

`rl/rollout.py` ve `rl/watch.py` her turn için şu loop'u uygular:

```text
available_pairs = (source_mask) & (planet_mask) & (src != tgt)
while launch_count < MAX_LAUNCHES_PER_PLAYER_TURN and any(source_mask):
    out = policy(obs_work)
    pair_logits = src_logits[..., :P, None] + tgt_logits     # (B, P, P)
    pair_logits[~available_pairs] = -inf
    ranked = top MAX_PAIR_CANDIDATES pairs by pair_logits
    for (src, tgt) in ranked:
        # Continuous Beta head: deterministic mean = α / (α + β)
        ship_value = α[src, tgt] / (α[src, tgt] + β[src, tgt])      # watch.py
        # (training rollout sample eder: ship_value ~ Beta(α, β))
        if decode_action_with_launch(src, tgt, ship_value) valid:
            launch -> mutate obs_work, disable (src, tgt) pair
            break
```

Sabitler `MAX_LAUNCHES_PER_PLAYER_TURN = 8`, `MAX_PAIR_CANDIDATES = 64`.

Watch (replay) deterministic Beta mean kullanır; training rollout stochastic sample kullanır.

In [13]:
from rl.watch import _act_all_sources, MAX_LAUNCHES_PER_PLAYER_TURN, MAX_PAIR_CANDIDATES
print('cap:', MAX_LAUNCHES_PER_PLAYER_TURN, '   pair_cap:', MAX_PAIR_CANDIDATES)

actions, infos = _act_all_sources(policy, raw_obs, 0, deterministic=True, device=torch.device('cpu'))
print('\nüretilen env action sayısı:', len(actions))
for info in infos[:8]:
    print(f'  src={info["src_slot"]:3d}  tgt={info["tgt_slot"]:3d}  '
          f'ship_value={info["ship_value"]:.3f}  value={info["value"]:+.3f}')

cap: 8    pair_cap: 64

üretilen env action sayısı: 1
  src= 12  tgt=  9  ship_value=0.513  value=+0.199
